# 🏆 India Book of Records — Build Your Own Generative AI Apps

**Maximum Students Simultaneously Building Generative AI Applications** · Sinthanai Academy × Sathyabama

Today you will build **two AI apps** in this one notebook:

| Part | App | What it does |
|---|---|---|
| **A** | 💬 Prompt-Template Chatbot | A chatbot with a personality *you* design |
| **B** | 📄 RAG Chatbot | A chatbot that answers questions **from a PDF** |

### How to run
1. Click the ▶ button on each cell, **from top to bottom**, one at a time.
2. Only edit the places marked ✏️ — everything else can stay as it is.
3. Take a screenshot at the 📸 marks.

> ⚠️ Your API key is like a password. Do **not** share this notebook after pasting your key.

---
# ⚙️ SETUP
## Step 1 — Install the libraries

In [1]:
!pip install -q openai pypdf faiss-cpu sentence-transformers
print("✅ Libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 21.6 MB/s eta 0:00:00
✅ Libraries installed!


## Step 2 — Choose your AI provider and paste your API key ✏️

You need **only ONE** key. Use whichever provider your trainer gave you (all are free).

| Provider | Get your free key here | Model used |
|---|---|---|
| `groq` | https://console.groq.com/keys | openai/gpt-oss-120b |
| `openrouter` | https://openrouter.ai/keys | nvidia/nemotron-3-ultra-550b-a55b:free |
| `nvidia` | https://build.nvidia.com (click *Get API Key*) | openai/gpt-oss-20b |
| `gemini` | https://aistudio.google.com/apikey | gemini-3-flash-preview |
| `cerebras` | https://cloud.cerebras.ai | gpt-oss-120b |

👉 Pick your provider from the dropdown on the right, paste your key in the box, then click ▶.

💡 **Why does one code work for all 5?** All five providers speak the same "OpenAI-style" language.
Only two things change: the **address** (`base_url`) and the **model name**.

In [3]:
# @title Step 2: Choose provider + paste API key
PROVIDER = "openrouter"  # @param ["groq", "openrouter", "nvidia", "gemini", "cerebras"]
API_KEY = ""  # @param {type:"string"}
MODEL = "openrouter/free" # @param {type:"string"}

# Each provider = an address (base_url) + a model name.
# Trainer tip: to change a model, just edit the "model" text below.
PROVIDERS = {
    "groq":       {"base_url": "https://api.groq.com/openai/v1"},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1"},   # any model ending in ":free" works
    "nvidia":     {"base_url": "https://integrate.api.nvidia.com/v1"},
    "gemini":     {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/"},
    "cerebras":   {"base_url": "https://api.cerebras.ai/v1"},
}

from openai import OpenAI

API_KEY = API_KEY.strip()
if API_KEY == "":
    raise ValueError("✋ Please paste your API key in the API_KEY box and run this cell again.")


client = OpenAI(api_key=API_KEY, base_url=PROVIDERS[PROVIDER]["base_url"])

print("✅ Settings saved")
print("   Provider :", PROVIDER)
print("   Model    :", MODEL)

✅ Settings saved
   Provider : openrouter
   Model    : openrouter/free


## Step 3 — The helper that talks to the AI + a quick test

`ask_llm(messages)` sends a list of messages to the AI and returns its reply.
If something goes wrong, it tells you **in plain English** what to fix.

In [4]:
def ask_llm(messages, temperature=0.7):
    """Send messages to the AI and return its reply as text."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,   # 0 = focused, 1 = creative
        )
        reply = response.choices[0].message.content
        return reply if reply else "(Empty reply — please run the cell again.)"

    except Exception as error:
        details = str(error)
        if "401" in details or "403" in details or "API key" in details:
            tip = "Your API key is wrong or inactive. Copy it again and re-run Step 2."
        elif "429" in details:
            tip = "Free limit reached for now. Wait 1 minute and try again."
        elif "404" in details:
            tip = "Model not found. Check the model name in Step 2."
        else:
            tip = "Something went wrong. Check your key / internet, or try another PROVIDER."
        return f"❌ {tip}\n   (Details: {details[:200]})"


# Quick test 👇
test = ask_llm([{"role": "user", "content": "Say hello to the students of the India Book of Records event in one line."}])
print("🤖", test)

🤖 Hello, bright minds of the India Book of Records event—keep inspiring the world!


---
# 💬 PART A — Prompt-Template Chatbot

Every chat with an AI is a list of **messages**. Each message has a **role**:

| Role | Who writes it | Purpose |
|---|---|---|
| `system` | You (the builder) | The bot's **personality and rules** — this comes from our prompt template |
| `user` | The person chatting | The question |
| `assistant` | The AI | The reply |

A **prompt template** is a sentence with blanks like `{bot_name}` and `{tone}`.
We fill the blanks with `.format()` to create the system prompt.

## A1 — Design your chatbot ✏️
Change the values in the boxes on the right, then click ▶.

In [5]:
# @title A1: Design your chatbot
BOT_NAME = "Study Buddy"  # @param {type:"string"}
ROLE = "a friendly study helper for engineering students"  # @param {type:"string"}
TONE = "cheerful and encouraging"  # @param {type:"string"}
LANGUAGE = "English"  # @param {type:"string"}
ANSWER_LENGTH = "3 to 4 short sentences"  # @param {type:"string"}

# 👇 The PROMPT TEMPLATE — words inside { } are blanks that get filled in.
PROMPT_TEMPLATE = """
You are {bot_name}, {role}.
Your tone is {tone}.
Always reply in {language}.
Keep every answer to {answer_length}.
If someone asks about something outside your role, politely say what you can help with instead.
"""

SYSTEM_PROMPT = PROMPT_TEMPLATE.format(
    bot_name=BOT_NAME,
    role=ROLE,
    tone=TONE,
    language=LANGUAGE,
    answer_length=ANSWER_LENGTH,
)

print("✅ Your chatbot's system prompt is:")
print(SYSTEM_PROMPT)

✅ Your chatbot's system prompt is:

You are Study Buddy, a friendly study helper for engineering students.
Your tone is cheerful and encouraging.
Always reply in English.
Keep every answer to 3 to 4 short sentences.
If someone asks about something outside your role, politely say what you can help with instead.



### 💡 Ideas — copy any row into the boxes above

| BOT_NAME | ROLE | TONE | LANGUAGE |
|---|---|---|---|
| Chef Amma | a home-cooking assistant who suggests simple South Indian recipes | warm and caring | English |
| Interview Coach | a mock interviewer who helps freshers prepare for placement interviews | professional but kind | English |
| Chennai Guide | a tour guide who knows Chennai's beaches, temples and food spots | fun and energetic | English |
| Tamil Teacher | a teacher who explains English words with their Tamil meaning | patient and simple | Tamil and English |
| Code Mentor | a Python tutor who explains code like talking to a beginner | calm and clear | English |
| Fitness Friend | a fitness coach who suggests easy home workouts for students | motivating | English |

### 🚀 Level 2 — edit the template itself
Add one of these lines inside `PROMPT_TEMPLATE` (before the closing `"""`) and run A1 again:
- `Always answer using bullet points.`
- `End every answer with one follow-up question to the user.`
- `Add one relevant emoji to every answer.`
- `End every answer with a short motivational quote.`

**Level 3 — add your own blank:** add a line `Your favourite topic is {topic}.` to the template,
create `TOPIC = "space science"` and add `topic=TOPIC` inside `.format(...)`.

## A2 — Ask one question ✏️

In [6]:
# @title A2: Ask your chatbot one question
QUESTION = "How can I prepare for my semester exams in 2 weeks?"  # @param {type:"string"}

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},   # personality (from template)
    {"role": "user",   "content": QUESTION},        # your question
]

print(f"🧑 You: {QUESTION}\n")
print(f"🤖 {BOT_NAME}: {ask_llm(messages)}")

🧑 You: How can I prepare for my semester exams in 2 weeks?

🤖 Study Buddy: Great that you’re starting early! First, make a realistic study schedule covering each subject, breaking topics into daily chunks. Next, gather past papers and focus on practice questions, timing yourself to simulate exam conditions. Finally, take short breaks, stay hydrated, and keep a positive mindset—your hard work will pay off!


## A3 — Chat continuously (the bot remembers the conversation)
Type your message in the box that appears. Type **exit** to stop.

📸 **Screenshot 1:** take a screenshot after 2–3 messages with your chatbot.

In [7]:
chat_history = [{"role": "system", "content": SYSTEM_PROMPT}]
print(f"🤖 {BOT_NAME} is ready! Type 'exit' to stop.\n")

while True:
    user_text = input("🧑 You: ")
    if user_text.strip().lower() in ["exit", "quit", "bye"]:
        print("👋 Chat ended.")
        break

    chat_history.append({"role": "user", "content": user_text})
    recent = chat_history[:1] + chat_history[-10:]      # system prompt + last 10 messages
    reply = ask_llm(recent)
    chat_history.append({"role": "assistant", "content": reply})

    print(f"\n🤖 {BOT_NAME}: {reply}\n")

🤖 Study Buddy is ready! Type 'exit' to stop.

🧑 You: hi

🤖 Study Buddy: Hi there! I'm Study Buddy, your friendly engineering study helper. How can I assist you today? Feel free to ask about concepts, problem‑solving tips, or anything else you need.

🧑 You: exit
👋 Chat ended.


---
# 📄 PART B — RAG Chatbot (chat with a PDF)

**RAG = Retrieval-Augmented Generation.** Think of an **open-book exam**:
instead of answering from memory, the AI first *looks up* the right pages in your PDF, then answers.

```
 PDF ──► Read text ──► Cut into small chunks ──► Turn chunks into numbers (embeddings) ──► Store in FAISS
                                                                                              │
 Answer ◄── AI writes the answer ◄── Find the 3 most similar chunks ◄──────────── Your question
```

Only **3 cells** to run: **B1** choose your PDF → **B2** build the RAG engine → **B3** ask questions.

## B1 — Paste your PDF link ✏️ (the ONLY thing you change in Part B)
- Leave it as it is to use the sample book on Neural Networks.
- Or paste the link of your PDF from the event library (your trainer will give it).
- Any direct PDF link works. A GitHub page link (`.../blob/...`) is also fine — it's converted automatically.

In [8]:
# @title B1: Paste your PDF link
PDF_URL = "https://github.com/Jaish19/India-Book-Of-Record/blob/main/neural_network.pdf"  # @param {type:"string"}

link = PDF_URL.strip()
if "github.com" in link and "/blob/" in link:                       # GitHub page link -> raw file link
    link = link.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

print("✅ PDF link saved. Now run B2.")

✅ PDF link saved. Now run B2.


## B2 — Build the RAG engine (just click ▶, no changes needed)
This one cell does all 5 RAG steps: **download → read → chunk → embed → store**.
Read the comments to see each step. ⏳ Takes 1–2 minutes for a big PDF.

In [9]:
import requests, faiss, numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# ---- Default settings (trainers can tweak; students need not touch) ----
CHUNK_SIZE    = 800    # characters per chunk
CHUNK_OVERLAP = 150    # characters repeated between chunks so sentences aren't lost
TOP_K         = 3      # how many chunks are given to the AI for each question

RAG_PROMPT_TEMPLATE = """
You are a helpful assistant who answers ONLY using the context below.
If the answer is not in the context, reply: "Sorry, I could not find this in the document."
Answer in 3 to 4 simple sentences.

Context:
{context}

Question: {question}

Answer:
"""

# ---- STEP 1: Download the PDF ----
link = PDF_URL.strip()
if "github.com" in link and "/blob/" in link:                       # GitHub page link -> raw file link
    link = link.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
response = requests.get(link, timeout=60)
if response.status_code != 200:
    raise ValueError(f"❌ Could not download the PDF (error {response.status_code}). Check the link in B1.")
with open("my_document.pdf", "wb") as f:
    f.write(response.content)
print("1️⃣  PDF downloaded")

# ---- STEP 2: Read the text, page by page ----
reader = PdfReader("my_document.pdf")
pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    if text.strip():
        pages.append({"page": page_number, "text": text})
if len(pages) == 0:
    raise ValueError("❌ No text found. This PDF may be a scanned image. Try another PDF.")
print(f"2️⃣  Read {len(pages)} pages")

# ---- STEP 3: Cut the text into small overlapping chunks ----
chunks = []
for p in pages:
    text = " ".join(p["text"].split())                               # clean extra spaces / new lines
    start = 0
    while start < len(text):
        chunks.append({"page": p["page"], "text": text[start : start + CHUNK_SIZE]})
        start = start + CHUNK_SIZE - CHUNK_OVERLAP
print(f"3️⃣  Created {len(chunks)} chunks")

# ---- STEP 4: Turn each chunk into 384 numbers (embeddings) ----
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True, show_progress_bar=True)
vectors = np.array(vectors, dtype="float32")
print("4️⃣  Embeddings created")

# ---- STEP 5: Store them in FAISS (a fast search engine for numbers) ----
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
print(f"5️⃣  Stored {index.ntotal} chunks in FAISS")

# ---- The RAG function: retrieve -> augment -> generate ----
def rag_ask(question):
    q_vector = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_vector, TOP_K)                       # 1. RETRIEVE similar chunks
    found = [chunks[i] for i in ids[0] if i != -1]
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in found)
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)        # 2. AUGMENT the prompt
    answer = ask_llm([{"role": "user", "content": prompt}], temperature=0.2)       # 3. GENERATE the answer
    return answer, sorted(set(c["page"] for c in found))

print("\n✅ RAG engine is ready! Now run B3 and ask questions.")

1️⃣  PDF downloaded
2️⃣  Read 223 pages
3️⃣  Created 999 chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

4️⃣  Embeddings created
5️⃣  Stored 999 chunks in FAISS

✅ RAG engine is ready! Now run B3 and ask questions.


## B3 — Ask your PDF questions
Type your question in the box. Type **exit** to stop.

Sample book questions: *What is a sigmoid neuron?* · *What is gradient descent?* · *What is the MNIST dataset?*
Also try a question that is **NOT** in the PDF (e.g. *Who won the 2011 Cricket World Cup?*) — a good RAG bot refuses to guess.

📸 **Screenshot 2:** take a screenshot after asking 2–3 questions about your PDF.

In [10]:
print("📄 Ask anything about your PDF. Type 'exit' to stop.\n")

while True:
    question = input("🧑 You: ")
    if question.strip().lower() in ["exit", "quit", "bye"]:
        print("👋 Done.")
        break
    answer, pages_used = rag_ask(question)
    print(f"\n🤖 {answer}\n   📄 Source pages: {pages_used}\n")

📄 Ask anything about your PDF. Type 'exit' to stop.

🧑 You: what is neuron

🤖 A neuron is a basic processing element in a neural network, also called a unit. It takes inputs and applies an activation function, such as s(z). Neurons are organized in layers, with inputs shown as vertical lines or as a 2‑D grid of values.
   📄 Source pages: [154, 178]

🧑 You: exit
👋 Done.


---
# 🎉 You built two Generative AI apps!

### 📸 Submission checklist
- [ ] Screenshot 1 — Part A chatbot conversation (A3)
- [ ] Screenshot 2 — Part B RAG answers with source pages (B3)

### 🛠️ Troubleshooting
| Message | Fix |
|---|---|
| `API key is wrong or inactive` | Copy the key again (no spaces), re-run Step 2 and Step 3 |
| `Free limit reached` | Wait 1 minute. Still failing? Ask your trainer for a key from another provider |
| `Model not found` | The model name in Step 2 is outdated — ask your trainer for the new one |
| `name 'client' is not defined` / `name 'rag_ask' is not defined` | You skipped a cell. Run all cells from the top, in order |
| `Could not download the PDF` | Check the link in B1 — it must open the PDF directly |
| `No text found` | The PDF is a scanned image — choose another PDF |
| Chat box stuck | Type `exit` and press Enter, or click ■ (stop) next to the cell |

### 🧪 Try next (optional)
- Changed your PDF? Just re-run B1 → B2 → B3.
- In B2, change `TOP_K` to 1 or 5 — do answers get better or worse?
- In B2, edit `RAG_PROMPT_TEMPLATE` so answers come as bullet points, or in Tamil.